In [1]:
import pandas as pd


In [3]:
# =========================
# LOAD RPA (1st innings)
# =========================
rpa_bb = pd.read_csv(
    "outputs/ball_by_ball_rpa_master.csv",
    usecols=[
        "match_id", "date", "league", 
        "innings",
        "batter", "bowler",
        "batting_team", "bowling_team",
        "rpa"
    ]
)

# =========================
# LOAD WPA (2nd innings)
# =========================
wpa_bb = pd.read_csv(
    "outputs/ball_by_ball_wp_wpa_t20.csv",
    usecols=[
        "match_id", "date", "league", "venue",
        "innings",
        "batter", "bowler",
        "batting_team", "bowling_team",
        "wpa"
    ]
)

rpa_bb["date"] = pd.to_datetime(rpa_bb["date"])
wpa_bb["date"] = pd.to_datetime(wpa_bb["date"])


In [4]:
# RPA only valid in 1st innings
rpa_bb = rpa_bb[rpa_bb["innings"] == 1].copy()

# WPA only valid in 2nd innings
wpa_bb = wpa_bb[wpa_bb["innings"] == 2].copy()


In [6]:
rpa_bat = (
    rpa_bb
    .groupby(
        ["match_id", "date", "league",
         "innings", "batter", "batting_team", "bowling_team"],
        as_index=False
    )["rpa"]
    .sum()
)

rpa_bat = rpa_bat.rename(columns={
    "batter": "player",
    "batting_team": "team",
    "bowling_team": "opponent"
})

rpa_bat["bat_value"] = rpa_bat["rpa"]
rpa_bat["bowl_value"] = 0.0


In [7]:
rpa_bowl = (
    rpa_bb
    .groupby(
        ["match_id", "date", "league",
         "innings", "bowler", "bowling_team", "batting_team"],
        as_index=False
    )["rpa"]
    .sum()
)

rpa_bowl = rpa_bowl.rename(columns={
    "bowler": "player",
    "bowling_team": "team",
    "batting_team": "opponent"
})

rpa_bowl["bat_value"] = 0.0
rpa_bowl["bowl_value"] = -rpa_bowl["rpa"]


In [8]:
wpa_bat = (
    wpa_bb
    .groupby(
        ["match_id", "date", "league", "venue",
         "innings", "batter", "batting_team", "bowling_team"],
        as_index=False
    )["wpa"]
    .sum()
)

wpa_bat = wpa_bat.rename(columns={
    "batter": "player",
    "batting_team": "team",
    "bowling_team": "opponent"
})

wpa_bat["bat_value"] = wpa_bat["wpa"]
wpa_bat["bowl_value"] = 0.0


In [9]:
wpa_bowl = (
    wpa_bb
    .groupby(
        ["match_id", "date", "league", "venue",
         "innings", "bowler", "bowling_team", "batting_team"],
        as_index=False
    )["wpa"]
    .sum()
)

wpa_bowl = wpa_bowl.rename(columns={
    "bowler": "player",
    "bowling_team": "team",
    "batting_team": "opponent"
})

wpa_bowl["bat_value"] = 0.0
wpa_bowl["bowl_value"] = -wpa_bowl["wpa"]


In [10]:
player_daily = pd.concat(
    [
        rpa_bat,
        rpa_bowl,
        wpa_bat,
        wpa_bowl
    ],
    ignore_index=True
)


In [11]:
player_daily = player_daily.drop(columns=["rpa", "wpa"], errors="ignore")


In [12]:
player_daily["innings_value"] = (
    player_daily["bat_value"] + player_daily["bowl_value"]
)


In [13]:
player_daily = (
    player_daily
    .sort_values(["player", "match_id", "innings"])
    .reset_index(drop=True)
)


In [14]:
(
    (player_daily["innings"] == 1) &
    ((player_daily["bat_value"] != 0) & (player_daily["bowl_value"] != 0))
).any()


np.False_

In [15]:
(
    (player_daily["innings"] == 1) &
    (player_daily["bat_value"].abs() > 0) &
    (player_daily["bat_value"].abs() < 0)
)


0         False
1         False
2         False
3         False
4         False
          ...  
159948    False
159949    False
159950    False
159951    False
159952    False
Length: 159953, dtype: bool

In [16]:
player_daily["innings_value"].describe()


count    1.599530e+05
mean     4.264509e-18
std      7.359635e+00
min     -6.535851e+01
25%     -9.659579e-01
50%     -5.524862e-03
75%      2.043463e-01
max      9.624132e+01
Name: innings_value, dtype: float64

In [17]:
player_daily[
    player_daily["player"] == "RG Sharma"
][["match_id", "innings", "bat_value", "bowl_value", "innings_value"]].head()


,match_id,innings,bat_value,bowl_value,innings_value
119165,682921,2,0.064776,0.0,0.064776
119166,682929,2,0.204402,0.0,0.204402
119167,682943,2,-0.062908,0.0,-0.062908
119168,682951,1,3.332714,0.0,3.332714
119169,682963,2,0.206892,0.0,0.206892


In [18]:
player_daily

,match_id,date,league,innings,player,team,opponent,bat_value,bowl_value,venue,innings_value
0,1157157,2019-02-25,SMAT,1,A Aravinddaraj,Puducherry,Uttar Pradesh,0.00000,-15.652720,NaN,-15.652720
1,1205739,2019-11-09,SMAT,2,A Aravinddaraj,Puducherry,Meghalaya,0.00000,-0.013592,Wankhede Stadium,-0.013592
2,1205757,2019-11-11,SMAT,2,A Aravinddaraj,Puducherry,Mizoram,0.00000,0.113694,BKC Ground,0.113694
3,1205783,2019-11-12,SMAT,1,A Aravinddaraj,Puducherry,Mumbai,0.00000,2.444613,NaN,2.444613
4,1205793,2019-11-14,SMAT,2,A Aravinddaraj,Puducherry,Haryana,0.00000,-0.000000,BKC Ground,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
159948,1076004,2017-02-25,PSL,1,Zulfiqar Babar,Quetta Gladiators,Peshawar Zalmi,-1.36749,0.000000,NaN,-1.367490
159949,1076004,2017-02-25,PSL,2,Zulfiqar Babar,Quetta Gladiators,Peshawar Zalmi,0.00000,0.121130,Dubai International Cricket Stadium,0.121130
159950,1076006,2017-02-28,PSL,2,Zulfiqar Babar,Quetta Gladiators,Peshawar Zalmi,0.00000,-0.030674,Sharjah Cricket Stadium,-0.030674
159951,1076009,2017-03-05,PSL,1,Zulfiqar Babar,Quetta Gladiators,Peshawar Zalmi,0.00000,3.730417,NaN,3.730417


In [19]:
player_daily["impact_type"] = player_daily["innings"].map(
    {1: "RPA", 2: "WPA"}
)


In [20]:
pd.crosstab(player_daily["innings"], player_daily["impact_type"])


impact_type,RPA,WPA
innings,,
1,81266,0
2,0,78687


In [22]:
venue_map = (
    wpa_bb[["match_id", "venue"]]
    .dropna()
    .drop_duplicates(subset="match_id")
)

player_daily = player_daily.merge(
    venue_map,
    on="match_id",
    how="left",
    suffixes=("", "_from_match")
)

player_daily["venue"] = player_daily["venue"].fillna(
    player_daily["venue_from_match"]
)

player_daily = player_daily.drop(columns=["venue_from_match"])



In [23]:
player_daily["venue"].isna().sum()


np.int64(0)

In [29]:
player_daily

,match_id,date,league,innings,player,team,opponent,bat_value,bowl_value,venue,innings_value,impact_type
0,1157157,2019-02-25,SMAT,1,A Aravinddaraj,Puducherry,Uttar Pradesh,0.00000,-15.652720,"Airforce Complex ground, Palam II",-15.652720,RPA
1,1205739,2019-11-09,SMAT,2,A Aravinddaraj,Puducherry,Meghalaya,0.00000,-0.013592,Wankhede Stadium,-0.013592,WPA
2,1205757,2019-11-11,SMAT,2,A Aravinddaraj,Puducherry,Mizoram,0.00000,0.113694,BKC Ground,0.113694,WPA
3,1205783,2019-11-12,SMAT,1,A Aravinddaraj,Puducherry,Mumbai,0.00000,2.444613,Wankhede Stadium,2.444613,RPA
4,1205793,2019-11-14,SMAT,2,A Aravinddaraj,Puducherry,Haryana,0.00000,-0.000000,BKC Ground,0.000000,WPA
...,...,...,...,...,...,...,...,...,...,...,...,...
159948,1076004,2017-02-25,PSL,1,Zulfiqar Babar,Quetta Gladiators,Peshawar Zalmi,-1.36749,0.000000,Dubai International Cricket Stadium,-1.367490,RPA
159949,1076004,2017-02-25,PSL,2,Zulfiqar Babar,Quetta Gladiators,Peshawar Zalmi,0.00000,0.121130,Dubai International Cricket Stadium,0.121130,WPA
159950,1076006,2017-02-28,PSL,2,Zulfiqar Babar,Quetta Gladiators,Peshawar Zalmi,0.00000,-0.030674,Sharjah Cricket Stadium,-0.030674,WPA
159951,1076009,2017-03-05,PSL,1,Zulfiqar Babar,Quetta Gladiators,Peshawar Zalmi,0.00000,3.730417,Gaddafi Stadium,3.730417,RPA


In [30]:
OUTPUT_PATH = "outputs/player_daily_innings_impact.csv"

player_daily.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved player_daily to: {OUTPUT_PATH}")


Saved player_daily to: outputs/player_daily_innings_impact.csv


In [40]:
import pandas as pd

def best_innings_performance(player_name, df):
    """
    Returns the player's best single-innings performance across all matches.
    """

    player_df = df[df["player"] == player_name].copy()

    if player_df.empty:
        return f"No data found for player: {player_name}"

    best_row = (
        player_df
        .sort_values("innings_value", ascending=False)
        .iloc[0]
    )

    return {
        "player": best_row["player"],
        "match_id": best_row["match_id"],
        "date": best_row["date"],
        "league": best_row["league"],
        "venue": best_row.get("venue", None),
        "innings": best_row["innings"],
        "impact_type": best_row["impact_type"],
        "bat_value": round(best_row["bat_value"], 3),
        "bowl_value": round(best_row["bowl_value"], 3),
        "innings_value": round(best_row["innings_value"], 3),
        "team": best_row["team"],
        "opponent": best_row["opponent"]
    }


In [47]:
df = pd.read_csv("outputs/player_daily_innings_impact.csv")

best_innings_performance("HH Pandya", df)


{'player': 'HH Pandya',
 'match_id': np.int64(966751),
 'date': '2016-02-27',
 'league': 'International',
 'venue': 'Shere Bangla National Stadium',
 'innings': np.int64(1),
 'impact_type': 'RPA',
 'bat_value': np.float64(0.0),
 'bowl_value': np.float64(30.578),
 'innings_value': np.float64(30.578),
 'team': 'India',
 'opponent': 'Pakistan'}

In [48]:
import pandas as pd
import numpy as np

def best_players_by_match(match_id, df):
    """
    Returns:
    - Best player in innings 1 (RPA)
    - Best player in innings 2 (WPA)
    - Best overall player (normalized within match)
    """

    match_df = df[df["match_id"] == match_id].copy()

    if match_df.empty:
        return f"No data found for match_id: {match_id}"

    results = {}

    # -------------------------
    # Best Innings 1 (RPA)
    # -------------------------
    inn1 = match_df[match_df["innings"] == 1]

    if not inn1.empty:
        best_inn1 = inn1.sort_values(
            "innings_value", ascending=False
        ).iloc[0]

        results["best_innings_1"] = {
            "player": best_inn1["player"],
            "innings_value": round(best_inn1["innings_value"], 3),
            "team": best_inn1["team"],
            "opponent": best_inn1["opponent"]
        }
    else:
        results["best_innings_1"] = None

    # -------------------------
    # Best Innings 2 (WPA)
    # -------------------------
    inn2 = match_df[match_df["innings"] == 2]

    if not inn2.empty:
        best_inn2 = inn2.sort_values(
            "innings_value", ascending=False
        ).iloc[0]

        results["best_innings_2"] = {
            "player": best_inn2["player"],
            "innings_value": round(best_inn2["innings_value"], 3),
            "team": best_inn2["team"],
            "opponent": best_inn2["opponent"]
        }
    else:
        results["best_innings_2"] = None

    # -------------------------
    # Normalized overall best
    # -------------------------
    match_df["norm_score"] = 0.0

    for inns in [1, 2]:
        sub = match_df[match_df["innings"] == inns]

        if len(sub) > 1:
            mean = sub["innings_value"].mean()
            std = sub["innings_value"].std()

            if std > 0:
                match_df.loc[sub.index, "norm_score"] = (
                    (sub["innings_value"] - mean) / std
                )

    best_overall = match_df.sort_values(
        "norm_score", ascending=False
    ).iloc[0]

    results["best_overall_normalized"] = {
        "player": best_overall["player"],
        "innings": int(best_overall["innings"]),
        "impact_type": best_overall["impact_type"],
        "raw_innings_value": round(best_overall["innings_value"], 3),
        "normalized_score": round(best_overall["norm_score"], 3),
        "team": best_overall["team"],
        "opponent": best_overall["opponent"]
    }

    return results


In [51]:
df = pd.read_csv("outputs/player_daily_innings_impact.csv")

best_players_by_match(1473507, df)


{'best_innings_1': {'player': 'RR Pant',
  'innings_value': np.float64(44.89),
  'team': 'Lucknow Super Giants',
  'opponent': 'Royal Challengers Bengaluru'},
 'best_innings_2': {'player': 'JM Sharma',
  'innings_value': np.float64(0.596),
  'team': 'Royal Challengers Bengaluru',
  'opponent': 'Lucknow Super Giants'},
 'best_overall_normalized': {'player': 'JM Sharma',
  'innings': 2,
  'impact_type': 'WPA',
  'raw_innings_value': np.float64(0.596),
  'normalized_score': np.float64(2.219),
  'team': 'Royal Challengers Bengaluru',
  'opponent': 'Lucknow Super Giants'}}